In [10]:
import torch
from torch import nn

net = nn.Sequential(
    nn.LazyLinear(8), # [2, 4] -> [2, 8]
    nn.ReLU(),        # [2, 8] -> [2, 8]
    nn.LazyLinear(1), # [2, 8] -> [2, 1]
)

# [2, 4]
X = torch.rand(
    size=(2, 4)
)

# Lazy parameter를 먼저 materialization.
Y = net(X)

print(Y.shape)

torch.Size([2, 1])


In [ ]:
# <class 'torch.nn.modules.linear.Linear'>
first_layer = net[0]
output_layer = net[2]

print("type(first_layer):", type(first_layer))
print("first_layer.weight.shape:", first_layer.weight.shape)
print("first_layer.bias.shape:", first_layer.bias.shape)

print("\ntype(output_layer):", type(output_layer))
print("output_layer.weight.shape", output_layer.weight.shape)
print("output_layer.bias.shape", output_layer.bias.shape)

assert isinstance(
    first_layer,
    nn.Linear,
)

assert isinstance(
    output_layer,
    nn.Linear,
)

type(first_layer): <class 'torch.nn.modules.linear.Linear'>
first_layer.weight.shape: torch.Size([8, 4])
first_layer.bias.shape: torch.Size([8])

type(output_layer): <class 'torch.nn.modules.linear.Linear'>
output_layer.weight.shape torch.Size([1, 8])
output_layer.bias.shape torch.Size([1])


In [12]:
# Normal weight & zero bias initialization

def init_normal(
    module: nn.Module,
):
    
    if isinstance(
        module,
        nn.Linear,
    ):
        
        nn.init.normal_(
            module.weight,
            mean=0,
            std=0.01,
        )
        
        nn.init.zeros_(
            module.bias
        )
        
net.apply(init_normal)

print(
    first_layer.weight.detach()[0]
)
print(
    first_layer.bias.detach()[0]
)

tensor([-0.0169,  0.0035,  0.0019,  0.0138])
tensor(0.)


In [13]:
# Constant initialization

def init_constant(
    module: nn.Module,
):
    
    if isinstance(
        module,
        nn.Linear,
    ):
        nn.init.constant_(
            module.weight,
            1, # fixed initialization value
        )
        nn.init.zeros_(
            module.bias
        )

net.apply(init_constant)

print(
    first_layer.weight.detach()[0]
)
print(
    first_layer.bias.detach()[0]
)

tensor([1., 1., 1., 1.])
tensor(0.)


In [14]:
# Layer마다 서로 다른 initialization 적용

def init_xavier(
    module: nn.Module,
):
    
    if isinstance(
        module,
        nn.Linear,
    ):
        # [Xavier Uniform Initialization]
        # weight 초기화 값을 uniform distribution에서 random하게 추출:
        # W(i,j) ~ U(-a, a), a = sqrt( 6 / ( N(in) + N(out)) )
        # 
        # - Neuron마다 다른 weight를 줘서 symmetry를 깬다
        # - Forward activation이 너무 커지거나 작아지는 것을 완화한다.
        # - Backward gradient가 exploding/vanishing하는 것도 완화한다.
        nn.init.xavier_uniform_(
            module.weight
        )
        
def init_42(
    module: nn.Module,
):
    
    if isinstance(
        module,
        nn.Linear,
    ):
        
        nn.init.constant_(
            module.weight,
            42, # fixed initialization value
        )
        
# 각 레이어에 서로 다른 initializer 적용
first_layer.apply(init_xavier)
output_layer.apply(init_42)


print(
    "first_layer.weight:",
    first_layer.weight.detach()[0]
)
print(
    "first_layer.bias:",
    first_layer.bias.detach()
)
print(
    "\noutput_layer.weight:",
    output_layer.weight.detach()
)
print(
    "output_layer.bias:",
    output_layer.bias.detach()
)

first_layer.weight: tensor([-0.4775, -0.3293, -0.2412, -0.4607])
first_layer.bias: tensor([0., 0., 0., 0., 0., 0., 0., 0.])

output_layer.weight: tensor([[42., 42., 42., 42., 42., 42., 42., 42.]])
output_layer.bias: tensor([0.])
